# Stacking: TF-IDF + BERTimbau (média de sementes, label smoothing) + XLM-R base + mDeBERTa (fp32)

**Modelos-base**
| Nome (cache) | Modelo | Parâmetros |
|---|---|---|
| `tfidf_word_lr` | TF-IDF palavras (1-2 gramas) + Regressão Logística | C=4 |
| `bertimbau_ls0p1_s0..s2` → `bertimbau_avg` | `neuralmind/bert-base-portuguese-cased`, **3 sementes por fold, média das probabilidades** | lr=2e-5, 3 épocas, warmup 0, 256 tokens, batch 16, **label smoothing 0.1** |
| `xlmr_base_lr2e5_ep3` | `FacebookAI/xlm-roberta-base` | lr=2e-5, 3 épocas, warmup 0.1 |
| `mdeberta_fp32_lr3e5_ep3` | `microsoft/mdeberta-v3-base` em **fp32** | lr=3e-5, 3 épocas, warmup 0.2 |

**Meta-modelo:** Regressão Logística sobre as log-probabilidades OOF (o BERTimbau entra como UMA feature: a média das sementes).

**Sobre o "otimizado" do BERTimbau (leia):** o tuning (22 variações, 1 semente cada) **não achou nenhum parâmetro com ganho acima do ruído**
(~0,7 ponto entre sementes). O que há aqui é a melhor evidência disponível:
- configuração-base `lr=2e-5`, 3 épocas (mais de 3 épocas **piora**: 5 épocas = −2 pontos);
- `label_smoothing=0.1`: maior acurácia do tuning (0,4628 contra média 0,4536 da base), mas **ainda não confirmado com várias sementes**;
- **média de várias sementes**: reduz o ruído entre sementes, que é do tamanho do ganho que se procura.
Compare `STACKING (LR)` com e sem o BERTimbau (ablação na seção 5) para ver quanto a média de sementes realmente ajudou.

**Tempo/unidades:** são 15 treinos de BERTimbau + 5 de XLM-R + 5 de mDeBERTa (o mais lento). Use `N_BERT_SEEDS`, `RUN_XLMR` e `RUN_MDEBERTA`
na célula de configuração para rodar só uma parte. Cada fold é salvo em cache (Drive), então dá para parar e retomar.
- Cada fold de transformer tem **proteção contra colapso** (repete com outra semente se prever quase uma classe só).


In [ ]:
!pip install -q transformers datasets accelerate openpyxl scikit-learn sentencepiece protobuf tqdm

In [ ]:
import gdown
file_id = "1iZt3Ja6vLiTbY7Gf5iFq0spDuyUHrTCC"
CAMINHO_ARQUIVO = "train.xlsx"
gdown.download(f"https://drive.google.com/uc?id={file_id}", CAMINHO_ARQUIVO, quiet=False)

In [ ]:
import gdown
#https://docs.google.com/spreadsheets/d/1eMrmvfSuNG6quc4v0yu-j3uxUdpOLsal/edit?usp=drive_link&ouid=100561416925788380508&rtpof=true&sd=true
file_id = "1eMrmvfSuNG6quc4v0yu-j3uxUdpOLsal"
CAMINHO_ARQUIVO = "test.xlsx"
gdown.download(f"https://drive.google.com/uc?id={file_id}", CAMINHO_ARQUIVO, quiet=False)

In [ ]:
import os, gc, math, warnings
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5

TRAIN_PATH = "/content/train.xlsx"
TEST_PATH = "/content/test.xlsx"
OUTPUT_PATH = "/content/drive/MyDrive/test1_rotulado_stacking_v4.xlsx"   # no Drive: o resultado final não se perde
# Cache no Drive: a máquina do Colab reinicia (troca de GPU, sessão que cai) e apaga /content e ../outputs.
# Se não quiser usar o Drive, troque por CACHE_DIR = "../outputs/stacking_cache".
from google.colab import drive
drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/stacking_cache"

RUN_TRANSFORMERS = True

# ---- BERTimbau ----
# O tuning NÃO achou parâmetro com ganho acima do ruído (veja a introdução do notebook). Aqui: config-base + label smoothing
# (maior acurácia do tuning, ainda não confirmada) + média de N_BERT_SEEDS sementes.
N_BERT_SEEDS = 3                 # 1 = sem média de sementes (mais rápido); cada semente = 5 treinos (um por fold)
BERT_LABEL_SMOOTHING = 0.1       # 0.0 desliga
BERTIMBAU_PARAMS = dict(epochs=3, lr=2e-5, warmup=0.0, bs=16, grad_acc=1, label_smoothing=BERT_LABEL_SMOOTHING)

RUN_XLMR = True
RUN_MDEBERTA = True              # o mais lento: ~2x o tempo de um BERT por treino (fp32)

_ls = str(BERT_LABEL_SMOOTHING).replace(".", "p")     # o nome do cache muda se você mudar o label smoothing
TRANSFORMERS = {}
for _i in range(N_BERT_SEEDS):
    TRANSFORMERS[f"bertimbau_ls{_ls}_s{_i}"] = dict(
        model_name="neuralmind/bert-base-portuguese-cased", seed_offset=100 * _i, **BERTIMBAU_PARAMS)
if RUN_XLMR:
    # XLM-R: lr=2e-5, 3 épocas foi o melhor; lr=3e-5 colapsou na triagem. Não suba o lr.
    TRANSFORMERS["xlmr_base_lr2e5_ep3"] = dict(model_name="FacebookAI/xlm-roberta-base",
                                               epochs=3, lr=2e-5, warmup=0.1, bs=16, grad_acc=1)
if RUN_MDEBERTA:
    # mDeBERTa: lr=3e-5 foi o melhor do tuning de 1 semente (0.455 contra 0.437 a 0.447 de lr=1e-5), ganho ainda não confirmado.
    # Se faltar memória, use bs=8, grad_acc=2 (mantém o batch efetivo de 16).
    TRANSFORMERS["mdeberta_fp32_lr3e5_ep3"] = dict(model_name="microsoft/mdeberta-v3-base", precision="fp32",
                                                   epochs=3, lr=3e-5, warmup=0.2, bs=16, grad_acc=1)

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Dados e folds

In [ ]:
label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

df = pd.read_excel(TRAIN_PATH)
df["resp_text"] = df["resp_text"].astype(str)
df["label"] = df["clarity"].map(label2id)
df["text_group"] = pd.factorize(df["resp_text"])[0]

test_df = pd.read_excel(TEST_PATH)
test_df["resp_text"] = test_df["resp_text"].astype(str)

y = df["label"].values

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(df, y, groups=df["text_group"]))

print("Treino:", df.shape, "| Teste:", test_df.shape)
print(df["clarity"].value_counts())
for k, (tr, va) in enumerate(folds):
    print(f"fold {k}: treino={len(tr)} val={len(va)}")

## 2. Utilitários (cache, relatório, barra de progresso)

In [ ]:
from tqdm.auto import tqdm

oof_preds, test_preds = {}, {}

# Barra geral: 1 unidade = 1 fold de 1 modelo-base (TF-IDF + transformers)
N_MODELS = 1 + (len(TRANSFORMERS) if RUN_TRANSFORMERS else 0)
overall = tqdm(total=N_MODELS * N_FOLDS, desc="Progresso geral", unit="fold",
               bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}{postfix}]")

def report(name, probs):
    pred = probs.argmax(1)
    tqdm.write(f"{name:<26} acc={accuracy_score(y, pred):.4f}  f1_macro={f1_score(y, pred, average='macro'):.4f}")

def load_cached(name):
    p_oof, p_test = f"{CACHE_DIR}/{name}_oof.npy", f"{CACHE_DIR}/{name}_test.npy"
    if os.path.exists(p_oof) and os.path.exists(p_test):
        return np.load(p_oof), np.load(p_test)
    return None

def register(name, oof, tst):
    np.save(f"{CACHE_DIR}/{name}_oof.npy", oof)
    np.save(f"{CACHE_DIR}/{name}_test.npy", tst)
    oof_preds[name], test_preds[name] = oof, tst
    report(name, oof)

def take(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def run_sklearn(name, make_model, X, X_test):
    cached = load_cached(name)
    if cached is not None:
        oof_preds[name], test_preds[name] = cached
        overall.update(N_FOLDS)
        report(name + " (cache)", cached[0])
        return
    oof = np.zeros((len(df), 3))
    tst = np.zeros((len(test_df), 3))
    for k, (tr, va) in enumerate(folds):
        overall.set_postfix_str(f"{name} | fold {k + 1}/{N_FOLDS}")
        model = make_model()
        model.fit(take(X, tr), y[tr])
        oof[va] = model.predict_proba(take(X, va))
        tst += model.predict_proba(X_test) / N_FOLDS
        overall.update(1)
    register(name, oof, tst)

## 3. Modelo 1: TF-IDF de palavras (1-2 gramas) + Regressão Logística

In [ ]:
run_sklearn(
    "tfidf_word_lr",
    lambda: Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", LogisticRegression(C=4, max_iter=2000)),
    ]),
    df["resp_text"], test_df["resp_text"],
)

## 4. Transformers: BERTimbau (várias sementes), XLM-R base e mDeBERTa (fine-tuning por fold)

- Número fixo de épocas, sem escolher checkpoint pelo fold de validação (senão o OOF fica otimista).
- **mDeBERTa em fp32:** em bf16 ele colapsou na triagem; em fp32 treinou bem (~13 min por treino na triagem).
- **Proteção contra colapso:** se um fold tiver acurácia < 0,38 ou prever > 85% numa só classe, ele é treinado de novo com outra semente (uma vez). O XLM-R colapsou na triagem com `lr` alto, então isso evita um fold estragar o OOF.
- Carga do modelo sempre em float32.

In [ ]:
import torch.nn.functional as F
from transformers import Trainer

class LSTrainer(Trainer):
    """Trainer com label smoothing (com label_smoothing=0 equivale à entropia cruzada padrão)."""
    label_smoothing = 0.0

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits.float(), labels, label_smoothing=self.label_smoothing)
        return (loss, outputs) if return_outputs else loss


def load_seqcls(name):
    """Carrega em float32 (alguns checkpoints são salvos em bf16 e quebram o treino se carregados assim)."""
    from transformers import AutoModelForSequenceClassification
    kw = dict(num_labels=3, label2id=label2id, id2label=id2label)
    for key in ("dtype", "torch_dtype"):          # o nome do argumento mudou entre versões
        try:
            return AutoModelForSequenceClassification.from_pretrained(name, **{key: torch.float32}, **kw)
        except (TypeError, ValueError):
            continue
    return AutoModelForSequenceClassification.from_pretrained(name, **kw)


def run_transformer(name, model_name, epochs=3, lr=2e-5, max_length=256, bs=16, grad_acc=1,
                    warmup=0.0, seed_offset=0, grad_ckpt=False, precision="auto", label_smoothing=0.0):
    cached = load_cached(name)
    if cached is not None:
        oof_preds[name], test_preds[name] = cached
        overall.update(N_FOLDS)
        report(name + " (cache)", cached[0])
        return

    from datasets import Dataset
    from transformers import AutoTokenizer, DataCollatorWithPadding, TrainingArguments, Trainer

    tok = AutoTokenizer.from_pretrained(model_name)
    tok_fn = lambda b: tok(b["resp_text"], truncation=True, max_length=max_length)
    collator = DataCollatorWithPadding(tokenizer=tok)

    def make_ds(frame, with_label=True):
        cols = ["resp_text", "label"] if with_label else ["resp_text"]
        return Dataset.from_pandas(frame[cols].reset_index(drop=True)).map(tok_fn, batched=True)

    # bf16 em GPU Ampere+ (A100/L4/G4/H100), fp16 nas demais; precision="fp32" força fp32
    has_cuda = torch.cuda.is_available()
    ampere = has_cuda and torch.cuda.get_device_capability()[0] >= 8
    use_bf16, use_fp16 = ampere, has_cuda and not ampere
    if precision == "fp32":
        use_bf16 = use_fp16 = False

    test_ds = make_ds(test_df, with_label=False)
    oof = np.zeros((len(df), 3))
    tst = np.zeros((len(test_df), 3))

    for k, (tr, va) in enumerate(folds):
        f_val, f_test = f"{CACHE_DIR}/{name}_fold{k}_val.npy", f"{CACHE_DIR}/{name}_fold{k}_test.npy"
        if os.path.exists(f_val) and os.path.exists(f_test):
            tqdm.write(f"[{name}] fold {k + 1}/{N_FOLDS}: carregado do cache")
            oof[va] = np.load(f_val); tst += np.load(f_test) / N_FOLDS
            overall.update(1)
            continue

        overall.set_postfix_str(f"{name} | fold {k + 1}/{N_FOLDS}")
        tqdm.write(f"[{name}] treinando fold {k + 1}/{N_FOLDS}")

        for attempt in range(2):                      # 2 tentativas: repete se o treino colapsar
            model = load_seqcls(model_name)
            total_steps = math.ceil(len(tr) / (bs * grad_acc)) * epochs
            args = TrainingArguments(
                output_dir=f"../models/stack_{name}_fold{k}",
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=16,
                gradient_accumulation_steps=grad_acc,
                num_train_epochs=epochs,
                weight_decay=0.01,
                warmup_steps=int(warmup * total_steps),
                gradient_checkpointing=grad_ckpt,
                save_strategy="no",
                fp16=use_fp16,
                bf16=use_bf16,
                logging_steps=100,
                report_to="none",
                disable_tqdm=False,
                seed=SEED + k + seed_offset + 1000 * attempt,
            )
            trainer = LSTrainer(model=model, args=args,
                                train_dataset=make_ds(df.iloc[tr]), data_collator=collator)
            trainer.label_smoothing = label_smoothing
            trainer.train()

            p_val = softmax(trainer.predict(make_ds(df.iloc[va])).predictions)
            p_test = softmax(trainer.predict(test_ds).predictions)

            pred_va = p_val.argmax(1)
            acc_k = float((pred_va == y[va]).mean())
            top_k = float(np.bincount(pred_va, minlength=3).max() / len(va))

            del trainer, model
            gc.collect(); torch.cuda.empty_cache()

            if acc_k >= 0.38 and top_k < 0.85:        # treino saudável
                break
            tqdm.write(f"[{name}] fold {k + 1}: possível colapso (acc={acc_k:.3f}, classe mais prevista={top_k:.2f})"
                       + (" -> repetindo com outra semente" if attempt == 0 else " (mantido)"))

        tqdm.write(f"[{name}] fold {k + 1}/{N_FOLDS}: acc={acc_k:.4f}")
        np.save(f_val, p_val); np.save(f_test, p_test)
        oof[va] = p_val; tst += p_test / N_FOLDS
        overall.update(1)

    register(name, oof, tst)


if RUN_TRANSFORMERS:
    for _name, _cfg in TRANSFORMERS.items():
        run_transformer(_name, **_cfg)

overall.set_postfix_str("modelos-base concluídos")
overall.close()

# ---- BERTimbau: média das sementes vira UM modelo-base (evita features muito correlacionadas no meta-modelo) ----
_members = sorted(n for n in oof_preds if n.startswith("bertimbau_ls"))
if _members:
    bert_members = {n: (oof_preds.pop(n), test_preds.pop(n)) for n in _members}
    for n in _members:
        report(n, bert_members[n][0])
    oof_preds["bertimbau_avg"] = np.mean([bert_members[n][0] for n in _members], axis=0)
    test_preds["bertimbau_avg"] = np.mean([bert_members[n][1] for n in _members], axis=0)
    report(f"bertimbau_avg ({len(_members)} sementes)", oof_preds["bertimbau_avg"])


## 5. Stacking: meta-modelo sobre as probabilidades OOF

In [ ]:
import itertools

names = list(oof_preds)
print("Modelos-base:", names)

groups = df["text_group"].values
META_SEEDS = (1, 2, 3)     # CVs agrupadas repetidas: reduzem o ruído da comparação

def feats(sel, store):
    # log-probabilidades: a LR do meta-modelo trabalha em escala de logits
    return np.hstack([np.log(np.clip(store[n], 1e-6, 1.0)) for n in sel])

def make_meta(C=1.0):
    return LogisticRegression(C=C, max_iter=3000)

def meta_oof(sel, C=1.0, seed=1):
    X = feats(sel, oof_preds)
    out = np.zeros((len(y), 3))
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)   # semente diferente da dos folds-base
    for tr, va in cv.split(X, y, groups=groups):
        out[va] = make_meta(C).fit(X[tr], y[tr]).predict_proba(X[va])
    return out

def meta_score(sel, C=1.0):
    accs, f1s = [], []
    for s in META_SEEDS:
        p = meta_oof(sel, C, s).argmax(1)
        accs.append(accuracy_score(y, p)); f1s.append(f1_score(y, p, average="macro"))
    return float(np.mean(accs)), float(np.mean(f1s))

print("\n=== Comparação (OOF no treino) ===")
for n in names:
    report(n, oof_preds[n])
report("média simples", np.mean([oof_preds[n] for n in names], axis=0))
a, f = meta_score(names)
print(f"{'STACKING (LR)':<26} acc={a:.4f}  f1_macro={f:.4f}   (média de {len(META_SEEDS)} CVs agrupadas)")

print("\n=== Ablação: stacking SEM cada modelo ===")
for n in names:
    a2, f2 = meta_score([m for m in names if m != n])
    print(f"sem {n:<22} acc={a2:.4f}  f1_macro={f2:.4f}   (variação: {a2 - a:+.4f})")

print("\n=== Concordância entre modelos (previsões OOF) ===")
P = {n: oof_preds[n].argmax(1) for n in names}
for m1, m2 in itertools.combinations(names, 2):
    print(f"{m1:<24} x {m2:<24} concordam={(P[m1] == P[m2]).mean():.3f}  ambos erram={((P[m1] != y) & (P[m2] != y)).mean():.3f}")

meta_pred = meta_oof(names, 1.0, 1).argmax(1)
print("\n", classification_report(y, meta_pred, target_names=["c1", "c234", "c5"]))
print("Matriz de confusão:\n", confusion_matrix(y, meta_pred))

## 6. Previsão final no teste

In [ ]:
X_all, X_test = feats(names, oof_preds), feats(names, test_preds)
meta_final = make_meta().fit(X_all, y)
test_probs = meta_final.predict_proba(X_test)
test_labels = np.array([id2label[i] for i in test_probs.argmax(1)])

submission_df = test_df.copy()
submission_df["clarity"] = test_labels
submission_df.to_excel(OUTPUT_PATH, index=False)

print("Salvo em:", OUTPUT_PATH)
print(pd.Series(test_labels).value_counts())
submission_df.head()